# Using Circumplex Instruments

Source: https://circumplex.jmgirard.com/articles/using-instruments.html

In [3]:
import circumplex
import pandas as pd

%load_ext autoreload
%autoreload 2

## Overview of Instrument-related Functions

Although the circumplex package is capable of analyzing and visualizing data in a "source-agnostic" manner (i.e., without knowing what the numbers correspond to), it can be helpful to both the user and the package to have more contextual information about which information/questionnaire the data come from. For example, knowing the specific instrument used can enable the package to automatically score item-level responses and standardize these scores using normative data. Furthermore, a centralized repository of information about circumplex instruments would provide a convenient and accessible way for users to discover and begin using new instruments.

The first part of this tutorial will discuss how to preview the instruments currently available in the circumplex package, how to load information about a specific instrument for use in analysis, and how to extract general and specific information about that instrument. The following functions will be discussed: `instruments()`, `instrument()`, `print()`, `summary()`, `scales()`, `items()`, `anchors()`, `norms()`, and `View()`.

The second part of this tutorial will discuss how to use the information about an instrument to transform and summarize circumplex data. It will demonstrate how to ipsatize item-level responses (i.e. apply deviation scoring across variables), how to calculate scale scores from item-level responses (with or without imputing/prorating missing values), and how to standardize scale scores using normative/comparison data. The following functions will be discussed: `ipsatize()`, `score()`, and `standardize()`.

## 2. Loading and Examining Instrument Objects

### Previewing the available instruments

You can preview the list of currently available instruments using the `instruments()` function. This function will print the abbreviation, name, and (in parentheses) the "code" for each available instrument. We will return to the code in the next section.

In [7]:
circumplex.instrument.instruments()

The circumplex package currently includes 4 instruments:
1. CSIP: Circumplex Scales of Interpersonal Problems (CSIP)
2. IIPSC: Inventory of Interpersonal Problems Short Circumplex (IIP-SC)
3. SSQP-eng: Swedish Soundscape Quality Protocol - English Translation (SSQP-eng)
4. SATP-eng: Soundscape Attributes Translation Project - English Translation (SATP-eng)


### Loading a specific instrument

To reduce loading time and memory usage, instrument information is not loaded into memory when the circumplex package is loaded. Instead, instruments should be loaded into memory on an as-needed bases. As demonstrated below, this can be done by passing an instrument's code (which we saw how to find in the last section) to the `load_instrument()` function. We can then examine that instrument data using the `print()` function.

In [18]:
csip = circumplex.instrument.load_instrument('CSIP')
print(csip)

CSIP: Circumplex Scales of Interpersonal Problems
64 Items, 8 Scales
Boudreaux, Ozer, Oltmanns, & Wright (2018)
<https://doi.org/10.1037/pas0000505>


### Examining an instrument in-depth

To examine the information available about a loaded instrument, there are several options. To print a long list of formatted information about the instrument, use the `summary()` function. This will return the same information returned by `print()`, followed by information about the instrument's scales, rating scale anchors, items, and normative data set(s). The summary of each instrument is also available from the package reference page.

In [19]:
csip.summary()

CSIP: Circumplex Scales of Interpersonal Problems
64 Items, 8 Scales
Boudreaux, Ozer, Oltmanns, & Wright (2018)
<https://doi.org/10.1037/pas0000505>

The CSIP contains 8 circumplex scales.
PA: Domineering (90°)
BC: Self-Centered (135°)
DE: Distant (180°)
FG: Socially Inhibited (225°)
HI: Nonassertive (270°)
JK: Exploitable (315°)
LM: Self-Sacrificing (360°)
NO: Intrusive (45°)

The CSIP is rated using the following 4-point scale.
0. Not a problem
1. Minor problem
2. Moderate problem
3. Serious problem

The CSIP contains 64 items (open-access).
1. Bossing around other people too much
2. Acting rude and inconsiderate toward others
3. Pushing away from other people who get too close
4. Difficulty making friends
5. Lacking self-confidence
6. Letting other people boss me around too much
7. Putting other people's needs before my own too much
8. Being overly affectionate with others
9. Verbally or physically abusing others
10. Acting selfishly with others
11. Difficulty showing love and affec

Specific subsections of this output can be returned individually by printing the `scales`, `anchors`, `items`, and `norms` attributes of the instrument object. These functions are especially useful when you need to know a specific bit of information about an instrument and don't want the console to be flooded with unneeded information. 

In [23]:
csip.anchors.show()

0. Not a problem
1. Minor problem
2. Moderate problem
3. Serious problem


Some of these attributes also have additional methods to customize their output. For instance, the `scales` attribute has a `.show()` method which includes the option to display the items for each scale.

In [27]:
csip.inst_items.show(n=5)

1. Bossing around other people too much
2. Acting rude and inconsiderate toward others
3. Pushing away from other people who get too close
4. Difficulty making friends
5. Lacking self-confidence

...and 59 more items.


In [25]:
csip.scales.show(inst_items=True)

PA: Domineering (90°)
	1: Bossing around other people too much
	9: Verbally or physically abusing others
	17: Starting arguments and conflicts with others
	25: Trying to influence or control other people too much
	33: Dominating or intimidating others
	41: Acting aggressively toward others
	49: Manipulating other people to get what I want
	57: Acting superior or condescending toward others
BC: Self-Centered (135°)
	2: Acting rude and inconsiderate toward others
	10: Acting selfishly with others
	18: Being unable to feel guilt or remorse
	26: Lacking respect for other people's beliefs, attitudes, or opinions
	34: Having trouble getting along with others
	42: Being insensitive to the thoughts, feelings, and needs of others
	50: Disliking most people
	58: Having trouble giving emotional or moral support to others
DE: Distant (180°)
	3: Pushing away from other people who get too close
	11: Difficulty showing love and affection to others
	19: Being unable to enjoy the company of others
	27:

## 3. Instrument-related tidying functions

It is a good idea in practice to digitize and save each participant's response to each item on an instrument, rather than just their scores on each scale. Having access to item-level data will make it easier to spot and correct mistakes, will enable more advanced analysis of missing data, and will enable latent variable models that account for measurement error (e.g. structural equation modelling). Furthermore, the functions described below will make it easy to transform and summarize such item-level data into scale scores.

First, however, we need to make sure the item-level data is in the expected format. Your data should be stored in a data frame where each row corresponds to one observation (e.g. participant, organization, or timepoint) and each column corresponds to one variable describing these observations (e.g. item responses, demographic characteristics, scale scores). The `pandas` package provides excellent tools for getting your data into this format from a variety of different file types and formats.

For the purpose of illustration, we will work with a small-scale data set, which includes item-level responses to the Inventory of Interpersonal Problems, Short Circumplex (IIP-SC) for just 10 participants. As will become important later on, this data set contains a small amount of missing values (represented as `NA`). This data set is included as part of the circumplex package and can be loaded and previewed as follows:


In [4]:
from circumplex.datasets import _raw_iipsc_path
raw_iipsc = pd.read_csv(_raw_iipsc_path)
print(raw_iipsc.head())

   IIP01  IIP02  IIP03  IIP04  IIP05  IIP06  IIP07  IIP08  IIP09  IIP10  ...  \
0      0      0      0    0.0      1      0      1      0      2      1  ...   
1      1      1      0    0.0      3      2      2      1      0      1  ...   
2      1      0      1    0.0      1      1      1      3      0      1  ...   
3      3      2      3    NaN      2      3      2      3      2      3  ...   
4      0      0      0    1.0      0      0      1      1      0      1  ...   

   IIP23  IIP24  IIP25  IIP26  IIP27  IIP28  IIP29  IIP30  IIP31  IIP32  
0      0      0      3      3      3      0      0      0      1      0  
1      0      0      0      0      0      1      0      0      0      2  
2      3      1      1      1      1      0      3      2      3      2  
3      3      2      1      2      3      2      3      2      3      2  
4      1      1      2      1      0      0      0      0      0      0  

[5 rows x 32 columns]


In [5]:
iipsc = circumplex.instrument.load_instrument('IIPSC')
iipsc.summary()

IIP-SC: Inventory of Interpersonal Problems Short Circumplex
32 Items, 8 Scales
Soldz, Budman, Demby, & Merry (1995)
<https://doi.org/10.1177/1073191195002001006>

The IIP-SC contains 8 circumplex scales.
PA: Domineering (90°)
BC: Vindictive (135°)
DE: Cold (180°)
FG: Socially Avoidant (225°)
HI: Nonassertive (270°)
JK: Exploitable (315°)
LM: Overly Nurturant (360°)
NO: Intrusive (45°)

The IIP-SC is rated using the following 5-point scale.
0. Not at all
1. Somewhat
2. Moderately
3. Very
4. Extremely

The IIP-SC contains 32 items (partial text).
1. ...point of view...
2. ...supportive of another...
3. ...show affection to...
4. ...join in on...
5. ...stop bothering me...
6. ...I am angry...
7. ...my own welfare...
8. ...keep things private...
9. ...too aggressive toward...
10. ...another person's happiness...
11. ...feeling of love...
12. ...introduce myself to...
13. ...confront people with...
14. ...assertive without worrying...
15. ...please other people...
16. ...open up to...
17. 

In [12]:
def score(data, instrument, na_rm=True, prorate=True):
    """Score item-level data using an instrument object.
    
    Args:
        data (pandas.DataFrame): A data frame where each row corresponds to one observation
            (e.g. participant, organization, timepoint) and each column corresponds to one variable
            describing these observations (e.g. item responses, demographic characteristics, scale scores).
        instrument (circumplex.instrument.Instrument): An instrument object loaded using the load_instrument() function.
        impute (bool): Whether to impute missing values before scoring.
        prorate (bool): Whether to prorate scale scores when missing values are present.
    
    Returns:
        pandas.DataFrame: A data frame where each row corresponds to one observation and each column
            corresponds to one variable describing these observations (e.g. item responses, demographic characteristics, scale scores).
    """
    # Step 1: Impute missing values
    # if impute:
        # data = circumplex.impute(data)
        
    # assert set(data.columns).issubset(instrument.inst_items.keys()), "Invalid item(s) selected."
            
    for i, scale in enumerate(instrument.scales.abbrev):
        items = list(instrument.scales.inst_items[i].keys())
        data[scale] = data.loc[:, items].mean(axis=1)
            
    return data

In [13]:
import numpy as np
raw_iipsc.columns = np.arange(1,len(raw_iipsc.columns)+1).astype(str)

score(raw_iipsc, iipsc)

,1,2,3,4,5,6,7,8,9,10,...,31,32,PA,BC,DE,FG,HI,JK,LM,NO
0,0,0,0,0.0,1,0,1,0,2,1,...,1,0,1.75,2.00,1.25,0.000000,0.50,0.250000,1.50,0.75
1,1,1,0,0.0,3,2,2,1,0,1,...,0,2,0.25,0.50,0.25,0.500000,2.00,1.750000,1.25,1.00
2,1,0,1,0.0,1,1,1,3,0,1,...,3,2,1.00,0.75,0.75,0.000000,2.25,2.000000,2.50,2.00
3,3,2,3,NaN,2,3,2,3,2,3,...,3,2,1.75,2.25,2.50,2.333333,2.50,2.000000,2.50,2.50
4,0,0,0,1.0,0,0,1,1,0,1,...,0,0,0.50,0.75,0.00,1.000000,0.50,0.250000,1.25,0.75
5,0,0,0,0.0,0,0,1,1,0,0,...,0,1,0.25,0.00,0.00,0.000000,0.00,0.000000,1.00,1.00
6,1,0,0,0.0,2,1,1,0,1,0,...,0,0,1.00,0.00,0.00,0.000000,1.00,1.000000,0.75,0.25
7,1,0,1,0.0,1,1,2,1,1,0,...,2,1,1.00,0.25,0.75,0.000000,0.50,0.666667,1.75,1.00
8,0,0,2,2.0,0,1,3,0,1,0,...,3,0,0.75,0.50,1.50,0.750000,0.00,1.000000,2.75,0.50
9,0,0,0,0.0,0,0,2,0,0,0,...,0,0,0.00,0.00,0.00,0.000000,0.00,0.500000,1.00,0.25


In [10]:
raw_iipsc[['1', '9', '17', '25']]


KeyError: "None of [Index(['1', '9', '17', '25'], dtype='object')] are in the [columns]"